# Sensitivity Analysis of Moment Scaling Exponents

This notebook quantifies the robustness of scaling exponents ζ(q) to phase picking uncertainties.

**Objective:** Test whether conclusions about multifractality and anomalous diffusion are sensitive to picking errors.

**Approach:**
1. Perturb P and S onset times with controlled noise/bias
2. Recalculate window segmentation and ζ(q) with perturbed picks
3. Compare perturbed ζ(q) against baseline via RMSE and correlation
4. Run Monte Carlo simulations for statistical confidence intervals

**Key questions:**
- Are RMSE values < 0.05 (robust) or > 0.15 (sensitive)?
- Does sensitivity vary across windows (pre-event, P, S, coda)?
- Do different coda methods show different robustness?

## 1. Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from pathlib import Path
import pickle
from src import (
    set_plot_style,
    analyze_all_windows,
    segment_all_signals,
    run_sensitivity_analysis,
    create_summary,
    plot_rmse_heatmap_by_coda,
    plot_zeta_confidence_by_coda,
    plot_rmse_heatmap_by_picking,
    plot_rmse_heatmap_by_datatype,
    load_sensitivity_results,
    load_baseline_results
)

colors, colors1 = set_plot_style()
logging.basicConfig(level=logging.WARNING)

print("Environment ready")

## 2. Configuration

### Perturbation Scenarios

**Gaussian noise:** Simulates random picking errors
- `noise_small`: σ = 0.2s (typical automatic picker precision)
- `noise_medium`: σ = 0.5s (moderate uncertainty)
- `noise_large`: σ = 1.0s (high uncertainty or poor SNR)

**Systematic bias:** Simulates picker calibration errors
- `bias_early`: -0.5s shift (picks too early)
- `bias_late`: +0.5s shift (picks too late)

**Monte Carlo:** 100 runs with σ = 0.5s for statistical aggregation

In [ ]:
# Dataset
DATASET = 'current'
PICKING_METHOD = 'ar_pick' # Options: 'ar_pick', 'phasenet'
DATA_TYPE = 'displacement'  # Options: 'acceleration', 'velocity', 'displacement'

# Coda methods to analyze
CODA_METHODS = ['rautian', 'arias', 'envelope', 'median']

# Perturbation scenarios
PERTURBATION_SCENARIOS = {
    'noise_small': {'type': 'gaussian', 'std': 0.2},
    'noise_medium': {'type': 'gaussian', 'std': 0.5},
    'noise_large': {'type': 'gaussian', 'std': 1.0},
    'bias_early': {'type': 'bias', 'bias': -0.5},
    'bias_late': {'type': 'bias', 'bias': 0.5}
}

# Monte Carlo parameters
N_MONTE_CARLO = 100
MONTE_CARLO_STD = 0.5

# Analysis parameters
TAU_MIN = 0.01
Q_VALUES = np.linspace(0.25, 5.0, 20)
SAMPLING_RATE = 200.0

# Signal columns
SIGNAL_COLUMN_MAP = {
    'acceleration': 'signal',
    'velocity': 'signal',
    'displacement': 'signal'
}

SIGNAL_COLUMN = SIGNAL_COLUMN_MAP[DATA_TYPE]

print(f"Configuration set for: {DATA_TYPE}")
print(f"Coda methods: {CODA_METHODS}")
print(f"Scenarios: {len(PERTURBATION_SCENARIOS)}")
print(f"Monte Carlo runs: {N_MONTE_CARLO}")

## 3. Load Data

Load:
- **Signals dictionary**: Original waveforms for all stations
- **Phase picks**: P and S onset times from PhaseNet
- **Baseline results**: ζ(q) computed with original (unperturbed) picks

Baseline format: DataFrame with columns `['window', 'q', 'zeta', 'zeta_err', 'r_squared', ...]`

In [ ]:
# Paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
if PICKING_METHOD == 'ar_pick':
    PICKS_DIR = DATA_DIR / f'03a_phase_identification_{PICKING_METHOD}' / DATA_TYPE
else:
    PICKS_DIR = DATA_DIR / f'03b_phase_identification_{PICKING_METHOD}' / DATA_TYPE

BASELINE_DIR = DATA_DIR / '04a_moment_scaling_spatial' / PICKING_METHOD / DATA_TYPE
OUTPUT_DIR = DATA_DIR / '05_sensitivity_analysis' / PICKING_METHOD / DATA_TYPE
FIGURES_DIR = PROJECT_ROOT / 'figures' / '05_sensitivity_analysis' / PICKING_METHOD / DATA_TYPE

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load signals
print(f"Loading data for: {DATA_TYPE}")
with open(PICKS_DIR / f'signals_{DATA_TYPE}_dict.pkl', 'rb') as f:
    signals_dict = pickle.load(f)
print(f"  Signals: {len(signals_dict)} stations")

# Load picks
df_full = pd.read_parquet(PICKS_DIR / f'df_full_{DATA_TYPE}_{PICKING_METHOD}.parquet')
print(f"  Picks: {len(df_full)} records")

# Load baseline results
baseline_results = {}
for coda_method in CODA_METHODS:
    baseline_file = BASELINE_DIR / coda_method / 'ensemble_spatial_summary.parquet'
    if baseline_file.exists():
        baseline_results[coda_method] = pd.read_parquet(baseline_file)
        print(f"  Baseline {coda_method}: loaded")

print("\nData loaded")

## 4. Run Sensitivity Analysis

For each coda method and perturbation scenario:
1. Perturb picks (apply noise or bias)
2. Recalculate window segmentation
3. Recompute ζ(q) via ensemble spatial scaling
4. Compare against baseline (RMSE, MAE, correlation)

**Monte Carlo:** Run 100 iterations with random seeds, aggregate statistics (mean ± std, percentiles).

**Output saved per scenario:**
- `ensemble_spatial_summary.parquet`: ζ(q) for all windows
- `ensemble_spatial_moments_{window}.parquet`: Full M_q(τ) data
- `monte_carlo_aggregated.parquet`: MC mean, std, p05, p95 over 100 runs

**Expected runtime:** ~30-60 min per data type (4 coda methods × 6 scenarios × 100 MC runs).

In [ ]:
# Pack configuration
config = {
    'TAU_MIN': TAU_MIN,
    'Q_VALUES': Q_VALUES,
    'SAMPLING_RATE': SAMPLING_RATE,
    'SIGNAL_COLUMN': SIGNAL_COLUMN,
    'N_MONTE_CARLO': N_MONTE_CARLO,
    'MONTE_CARLO_STD': MONTE_CARLO_STD
}

# Run analysis
results = run_sensitivity_analysis(
    data_type=DATA_TYPE,
    signals_dict=signals_dict,
    df_full=df_full,
    baseline_results=baseline_results,
    coda_methods=CODA_METHODS,
    perturbation_scenarios=PERTURBATION_SCENARIOS,
    segment_function=segment_all_signals,
    analyze_function=analyze_all_windows,
    config=config,
    output_dir=OUTPUT_DIR,
    verbose=False
)

## 5. Save results

In [ ]:
# Save complete results
results_file = OUTPUT_DIR / f'sensitivity_results_{DATA_TYPE}.pkl'
with open(results_file, 'wb') as f:
    pickle.dump(results, f)
print(f"Saved complete results: {results_file}")

# Create and save summary
df_summary = create_summary(
    results,
    data_type=DATA_TYPE,
    save_path=OUTPUT_DIR / f'sensitivity_summary_{DATA_TYPE}.csv'
)

print(f"Summary shape: {df_summary.shape}")
print(f"Saved summary: {OUTPUT_DIR / f'sensitivity_summary_{DATA_TYPE}.csv'}")

## 6. View summary

In [ ]:
# Display summary pivot table
print(f"\nSUMMARY: {DATA_TYPE.upper()}")
print("="*80)

summary_pivot = df_summary.pivot_table(
    index=['scenario', 'window'],
    columns='coda_method',
    values='rmse',
    aggfunc='mean'
)

print(summary_pivot)

# Top 10 worst cases
print(f"\n\nTOP 10 WORST CASES (highest RMSE)")
print("="*80)

worst_cases = df_summary.nlargest(10, 'rmse')[[
    'coda_method', 'scenario', 'window', 'rmse', 'mae', 'correlation'
]]
print(worst_cases.to_string(index=False))

## 7. Visualization

Structure:
1. Section to run AFTER run_sensitivity_analysis()
2. Section to run AT THE END (comparative, after all data types)

### 7.1 Compare Coda Methods for each data type

**Purpose:** Compare coda detection methods for the current data type (acceleration/velocity/displacement).

**Generated plots:**
1. **RMSE Heatmap** (2×2 grid) — shows sensitivity of each coda method to different perturbation scenarios
2. **ζ(q) Confidence Bands** (2×2 grid) — shows baseline scaling exponents with Monte Carlo uncertainty intervals

**Output location:** `figures/05_sensitivity_analysis/{picking_method}/{data_type}

 **Run this section every time you execute the notebook for a new data type.**


In [ ]:
# Plot 1: RMSE heatmap (2×2 grid, one per coda method)
plot_rmse_heatmap_by_coda(
    results=results,
    data_type=DATA_TYPE,
    picking_method=PICKING_METHOD,
    save_dir=FIGURES_DIR
)
 
# Plot 2: ζ(q) with Monte Carlo confidence bands
plot_zeta_confidence_by_coda(
    results=results,
    baseline_results=baseline_results,
    data_type=DATA_TYPE,
    picking_method=PICKING_METHOD,
    q_values=Q_VALUES,
    save_dir=FIGURES_DIR
)

### Comparative plots

 **IMPORTANT: Run this section ONLY ONCE, after completing sensitivity analysis for ALL three data types.**

### Prerequisites:
Before running this section, ensure you have executed the entire notebook (including Section 1) for:
- `acceleration`
- `velocity`  
- `displacement`

And for both picking methods:
- `ar_pick`
- `phasenet`

### What this section does:

- Compare picking methods (AR-AIC vs PhaseNet)
- Total: **12 plots** (3 data types × 4 coda methods)

- Compare data types (acceleration/velocity/displacement)
- Total: **8 plots** (2 picking methods × 4 coda methods)

**Output location:** `data/processed/05_sensitivity_analysis/comparison_plots/`

In [ ]:
COMPARISON_PLOTS_DIR = FIGURES_DIR = PROJECT_ROOT / 'figures' / '05_sensitivity_analysis' / 'comparison_plots'
COMPARISON_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
 
DATA_TYPES = ['acceleration', 'velocity', 'displacement']
PICKING_METHODS = ['ar_pick', 'phasenet']
CODA_METHODS = ['rautian', 'arias', 'envelope', 'median']

In [ ]:
# Structure: {data_type: {picking_method: results}}
all_results = {}
 
for data_type in DATA_TYPES:
    all_results[data_type] = {}
    
    for picking_method in PICKING_METHODS:
        
        try:
            results = load_sensitivity_results(DATA_DIR, data_type, picking_method)
            all_results[data_type][picking_method] = results
            print(f"  ✓ {data_type}/{picking_method}: {len(results)} coda methods")
            
        except FileNotFoundError as e:
            print(f"  ✗ {data_type}/{picking_method}: NOT FOUND")
            all_results[data_type][picking_method] = None
 
print("\n")

In [ ]:
dir1= COMPARISON_PLOTS_DIR / 'picking_comparison'
n_plots_b = 0
 
for data_type in DATA_TYPES:
    
    # Check if both picking methods are available
    if all_results[data_type]['ar_pick'] is None or \
       all_results[data_type]['phasenet'] is None:
        print(f"  Skipping {data_type}: missing picking method results")
        continue
    
    for coda_method in CODA_METHODS:
        
        # Prepare results dict for this comparison
        results_by_picking = {
            'ar_pick': all_results[data_type]['ar_pick'],
            'phasenet': all_results[data_type]['phasenet']
        }
        
        try:
            plot_rmse_heatmap_by_picking(
                results_dict=results_by_picking,
                data_type=data_type,
                coda_method=coda_method,
                save_dir=dir1
            )
            n_plots_b += 1
            
        except Exception as e:
            print(f"Error: {data_type}/{coda_method}: {e}")

 

In [ ]:
dir2 = COMPARISON_PLOTS_DIR / 'datatype_comparison'
n_plots_c = 0
 
for picking_method in PICKING_METHODS:
    
    # Check if all data types are available for this picking method
    results_available = {}
    all_available = True
    
    for data_type in DATA_TYPES:
        if all_results[data_type][picking_method] is None:
            all_available = False
            print(f"  Skipping {picking_method}: missing {data_type}")
            break
        results_available[data_type] = all_results[data_type][picking_method]
    
    if not all_available:
        continue
    
    for coda_method in CODA_METHODS:
        
        try:
            plot_rmse_heatmap_by_datatype(
                results_dict=results_available,
                picking_method=picking_method,
                coda_method=coda_method,
                save_dir=dir2
            )
            n_plots_c += 1
            
        except Exception as e:
            print(f"Error: {picking_method}/{coda_method}: {e}")
 